# 02. 버킷 엔진 — 인출 시뮬레이션 & 가드레일 인출법

이 Notebook에서 하는 일:
1. 버킷별 상세 현황 분석
2. 가드레일 인출법 (Guardrail Withdrawal) 계산
3. 버킷 충전 로직 (버킷 1 → 보충 시점 판단)
4. 인출 지속 가능 기간 시뮬레이션
5. 인출 이력 DB 저장

In [1]:
# 공통 설정
import pandas as pd
import numpy as np
import yaml, sqlite3, os
from pathlib import Path
from dotenv import load_dotenv
from datetime import date, datetime

load_dotenv()
PROJECT_ROOT = Path.cwd()

with open(PROJECT_ROOT / 'config.yaml', encoding='utf-8') as f:
    CONFIG = yaml.safe_load(f)

MONTHLY_EXPENSE = CONFIG['user']['monthly_expense']
TARGET          = CONFIG['portfolio']
WITHDRAWAL_CFG  = CONFIG['withdrawal']

print(f"사용자: {CONFIG['user']['name']}")
print(f"월 생활비: {MONTHLY_EXPENSE:,}원")
print(f"기본 인출률: {WITHDRAWAL_CFG['base_rate']*100:.1f}%")

사용자: 종헌
월 생활비: 2,500,000원
기본 인출률: 4.0%


In [2]:
# 자산 데이터 로드
assets_path = PROJECT_ROOT / 'assets.csv'
sample_path = PROJECT_ROOT / 'assets_sample.csv'

if assets_path.exists():
    df = pd.read_csv(assets_path, encoding='utf-8-sig')
else:
    df = pd.read_csv(sample_path, encoding='utf-8-sig')
    print('※ 샘플 데이터 사용 중')

mask = df['current_value'].isna() | (df['current_value'] == 0)
df.loc[mask, 'current_value'] = df.loc[mask, 'quantity'] * df.loc[mask, 'unit_price']
df['current_value'] = pd.to_numeric(df['current_value'], errors='coerce').fillna(0)

BUCKET_MAP = {'cash': 1, 'bond': 2, 'tdf': 2, 'equity': 3, 'income': 3}
df['bucket'] = df['asset_type'].str.lower().map(BUCKET_MAP).fillna(0).astype(int)

bucket_sum = df.groupby('bucket')['current_value'].sum()
total = df['current_value'].sum()
b1 = bucket_sum.get(1, 0)
b2 = bucket_sum.get(2, 0)
b3 = bucket_sum.get(3, 0)

print(f'총 자산: {total:,.0f}원')
print(f'버킷 1 (현금성): {b1:,.0f}원')
print(f'버킷 2 (채권):   {b2:,.0f}원')
print(f'버킷 3 (성장):   {b3:,.0f}원')

※ 샘플 데이터 사용 중
총 자산: 67,550,000원
버킷 1 (현금성): 35,000,000원
버킷 2 (채권):   31,130,000원
버킷 3 (성장):   1,420,000원


## 1. 버킷별 상세 현황

In [3]:
ANNUAL_EXPENSE = MONTHLY_EXPENSE * 12

months_b1 = b1 / MONTHLY_EXPENSE
years_total = total / ANNUAL_EXPENSE

print('=' * 50)
print('  버킷별 상세 현황')
print('=' * 50)
print()

bucket_info = [
    ('버킷 1', '현금성 (즉시 인출)', b1, '예금·CMA·MMF'),
    ('버킷 2', '중기 안정 (채권/TDF)', b2, '채권·단기채ETF·TDF'),
    ('버킷 3', '장기 성장 (주식/인컴)', b3, '주식형ETF·리츠·배당'),
]

for name, desc, amount, assets in bucket_info:
    pct = amount / total * 100 if total > 0 else 0
    months = amount / MONTHLY_EXPENSE if MONTHLY_EXPENSE > 0 else 0
    print(f'{name}: {desc}')
    print(f'  금액: {amount:>15,.0f}원  ({pct:.1f}%)')
    print(f'  생활비 {months:.1f}개월치')
    print(f'  포함 자산: {assets}')
    print()

print(f'{'─'*50}')
print(f'총 자산: {total:,.0f}원 → 연 생활비 {ANNUAL_EXPENSE:,.0f}원 기준 {years_total:.1f}년치')

  버킷별 상세 현황

버킷 1: 현금성 (즉시 인출)
  금액:      35,000,000원  (51.8%)
  생활비 14.0개월치
  포함 자산: 예금·CMA·MMF

버킷 2: 중기 안정 (채권/TDF)
  금액:      31,130,000원  (46.1%)
  생활비 12.5개월치
  포함 자산: 채권·단기채ETF·TDF

버킷 3: 장기 성장 (주식/인컴)
  금액:       1,420,000원  (2.1%)
  생활비 0.6개월치
  포함 자산: 주식형ETF·리츠·배당

──────────────────────────────────────────────────
총 자산: 67,550,000원 → 연 생활비 30,000,000원 기준 2.3년치


## 2. 가드레일 인출법 (Guardrail Withdrawal)

- **기본 인출률**: 연 4% (월 생활비 기준)
- **상향 가드레일**: 포트폴리오 +20% 이상 → 인출 5%로 증액 가능
- **하향 가드레일**: 포트폴리오 -20% 이하 → 인출 3%로 감액 권고

In [4]:
base_rate   = WITHDRAWAL_CFG['base_rate']     # 0.04
upper_guard = WITHDRAWAL_CFG['upper_guard']   # 0.20
lower_guard = WITHDRAWAL_CFG['lower_guard']   # -0.20

# DB에서 1년 전 총 자산 조회 (없으면 현재 값 사용)
db_path = PROJECT_ROOT / 'portfolio.db'
conn = sqlite3.connect(db_path)
cur  = conn.cursor()

cur.execute('''
    SELECT total FROM bucket_snapshots
    WHERE date <= date('now', '-365 days')
    ORDER BY date DESC LIMIT 1
''')
row = cur.fetchone()
conn.close()

total_1y_ago = row[0] if row else total  # 1년 전 자산 (없으면 현재값)
portfolio_return = (total - total_1y_ago) / total_1y_ago if total_1y_ago > 0 else 0

# 가드레일 판정
if portfolio_return >= upper_guard:
    actual_rate = base_rate + 0.01  # 5%
    guardrail_status = f'🟢 상향 가드레일 적용 (수익률 {portfolio_return*100:+.1f}%) → 인출률 {actual_rate*100:.0f}%'
    guardrail_applied = 0
elif portfolio_return <= lower_guard:
    actual_rate = base_rate - 0.01  # 3%
    guardrail_status = f'🔴 하향 가드레일 적용 (수익률 {portfolio_return*100:+.1f}%) → 인출률 {actual_rate*100:.0f}%'
    guardrail_applied = 1
else:
    actual_rate = base_rate  # 4%
    guardrail_status = f'🟡 기본 인출률 유지 (수익률 {portfolio_return*100:+.1f}%) → 인출률 {actual_rate*100:.0f}%'
    guardrail_applied = 0

annual_withdrawal  = total * actual_rate
monthly_withdrawal = annual_withdrawal / 12

print('=== 가드레일 인출법 계산 결과 ===')
print()
print(f'현재 총 자산:      {total:>15,.0f}원')
print(f'1년 전 총 자산:    {total_1y_ago:>15,.0f}원')
print(f'포트폴리오 수익률:  {portfolio_return*100:+.1f}%')
print()
print(f'가드레일 판정: {guardrail_status}')
print()
print(f'권장 연간 인출액:  {annual_withdrawal:>15,.0f}원')
print(f'권장 월 인출액:    {monthly_withdrawal:>15,.0f}원')
print(f'현재 월 생활비:    {MONTHLY_EXPENSE:>15,.0f}원')
print()

gap = monthly_withdrawal - MONTHLY_EXPENSE
if gap >= 0:
    print(f'✅ 인출 가능액이 생활비보다 {gap:,.0f}원 많습니다.')
else:
    print(f'⚠️ 생활비가 인출 가능액보다 {abs(gap):,.0f}원 많습니다. 지출 조정을 검토하세요.')

=== 가드레일 인출법 계산 결과 ===

현재 총 자산:           67,550,000원
1년 전 총 자산:         67,550,000원
포트폴리오 수익률:  +0.0%

가드레일 판정: 🟡 기본 인출률 유지 (수익률 +0.0%) → 인출률 4%

권장 연간 인출액:        2,702,000원
권장 월 인출액:            225,167원
현재 월 생활비:          2,500,000원

⚠️ 생활비가 인출 가능액보다 2,274,833원 많습니다. 지출 조정을 검토하세요.


## 3. 버킷 충전 로직

버킷 1(현금성)이 **6개월치** 미만이 되면 버킷 2에서 보충합니다.  
버킷 2가 부족하면 버킷 3에서 보충합니다.

In [5]:
REFILL_THRESHOLD_MONTHS = 6   # 버킷 1이 6개월치 미만이면 보충
TARGET_MONTHS           = 12  # 12개월치까지 채우기

refill_target = TARGET_MONTHS * MONTHLY_EXPENSE
refill_needed = max(0, refill_target - b1)
months_now    = b1 / MONTHLY_EXPENSE

print('=== 버킷 충전 로직 ===')
print()
print(f'버킷 1 현재:    {b1:>15,.0f}원 ({months_now:.1f}개월치)')
print(f'충전 기준:      {REFILL_THRESHOLD_MONTHS}개월치 미만이면 보충')
print(f'목표 수준:      {TARGET_MONTHS}개월치 ({refill_target:,.0f}원)')
print()

if months_now < REFILL_THRESHOLD_MONTHS:
    print(f'🔴 버킷 1 보충 필요: {refill_needed:,.0f}원 부족')
    print()

    # 버킷 2에서 보충 가능 여부
    b2_min = 6 * MONTHLY_EXPENSE  # 버킷 2 최소 유지분
    b2_available = max(0, b2 - b2_min)

    if b2_available >= refill_needed:
        print(f'  → 버킷 2에서 {refill_needed:,.0f}원 이동 권장')
        print(f'     (버킷 2 잔액: {b2:,.0f}원 → 이동 후 {b2-refill_needed:,.0f}원)')
    else:
        from_b2 = b2_available
        from_b3 = refill_needed - from_b2
        print(f'  → 버킷 2에서 {from_b2:,.0f}원 + 버킷 3에서 {from_b3:,.0f}원 이동 권장')
        print(f'     ⚠️ 버킷 3(성장 자산) 매도 필요 — 시장 상황 확인 후 진행하세요')
else:
    print(f'✅ 버킷 1 충전 불필요 ({months_now:.1f}개월치 확보됨)')
    print(f'   다음 충전 기준: {months_now - REFILL_THRESHOLD_MONTHS:.1f}개월 후 점검')

=== 버킷 충전 로직 ===

버킷 1 현재:         35,000,000원 (14.0개월치)
충전 기준:      6개월치 미만이면 보충
목표 수준:      12개월치 (30,000,000원)

✅ 버킷 1 충전 불필요 (14.0개월치 확보됨)
   다음 충전 기준: 8.0개월 후 점검


## 4. 인출 지속 가능 기간 시뮬레이션

현재 자산으로 몇 년간 인출이 가능한지 시뮬레이션합니다.  
가정: 버킷 3 연평균 수익률 5%, 물가상승률 2.5%

In [6]:
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = 'iframe'

inflation_rate = CONFIG['inflation']['assumed_rate']   # 0.025
growth_rate_b3 = 0.05   # 버킷 3 연평균 수익률 가정
growth_rate_b2 = 0.025  # 버킷 2 연평균 수익률 가정
sim_years      = 40     # 시뮬레이션 기간

# 시뮬레이션
sim_b1, sim_b2, sim_b3 = b1, b2, b3
sim_expense = MONTHLY_EXPENSE * 12

years_list   = [0]
b1_list      = [sim_b1]
b2_list      = [sim_b2]
b3_list      = [sim_b3]
total_list   = [sim_b1 + sim_b2 + sim_b3]
bankrupt_year = None

for yr in range(1, sim_years + 1):
    # 연간 인출 (물가 반영)
    sim_expense *= (1 + inflation_rate)

    # 버킷 3, 2 수익 적용
    sim_b3 *= (1 + growth_rate_b3)
    sim_b2 *= (1 + growth_rate_b2)

    # 버킷 1에서 인출
    sim_b1 -= sim_expense

    # 버킷 1이 6개월치 미만이면 버킷 2에서 보충
    if sim_b1 < 6 * (sim_expense / 12):
        refill = min(sim_b2 * 0.3, max(0, 12 * (sim_expense / 12) - sim_b1))
        sim_b1 += refill
        sim_b2 -= refill

    # 버킷 2가 3개월치 미만이면 버킷 3에서 보충
    if sim_b2 < 3 * (sim_expense / 12):
        refill = min(sim_b3 * 0.2, max(0, 12 * (sim_expense / 12) - sim_b2))
        sim_b2 += refill
        sim_b3 -= refill

    sim_b1 = max(0, sim_b1)
    sim_b2 = max(0, sim_b2)
    sim_b3 = max(0, sim_b3)
    sim_total = sim_b1 + sim_b2 + sim_b3

    years_list.append(yr)
    b1_list.append(sim_b1)
    b2_list.append(sim_b2)
    b3_list.append(sim_b3)
    total_list.append(sim_total)

    if sim_total <= 0 and bankrupt_year is None:
        bankrupt_year = yr
        break

# 차트
fig = go.Figure()
fig.add_trace(go.Scatter(x=years_list, y=[v/1e8 for v in b1_list],
    name='버킷 1 (현금)', fill='tozeroy', line=dict(color='#00B0F0')))
fig.add_trace(go.Scatter(x=years_list, y=[v/1e8 for v in b2_list],
    name='버킷 2 (채권)', fill='tozeroy', line=dict(color='#70AD47')))
fig.add_trace(go.Scatter(x=years_list, y=[v/1e8 for v in b3_list],
    name='버킷 3 (성장)', fill='tozeroy', line=dict(color='#FF6B35')))
fig.add_trace(go.Scatter(x=years_list, y=[v/1e8 for v in total_list],
    name='총 자산', line=dict(color='#1F3864', width=3, dash='dot')))

fig.update_layout(
    title=f'인출 지속 가능 시뮬레이션 (물가상승률 {inflation_rate*100:.1f}%, 버킷3 수익률 {growth_rate_b3*100:.0f}%)',
    xaxis_title='경과 연도',
    yaxis_title='자산 (억원)',
    height=450,
    font=dict(size=13)
)
fig.show()

print()
if bankrupt_year:
    print(f'⚠️ 시뮬레이션 결과: {bankrupt_year}년 후 자산 소진 예상')
else:
    final_total = total_list[-1]
    print(f'✅ {sim_years}년 후 잔여 자산: {final_total:,.0f}원 ({final_total/1e8:.1f}억원)')
    print(f'   → {sim_years}년간 인출 지속 가능합니다.')


✅ 40년 후 잔여 자산: 7,085원 (0.0억원)
   → 40년간 인출 지속 가능합니다.


## 5. 이번 달 인출 계획 확정 및 DB 저장

In [7]:
today = str(date.today())

print('=== 이번 달 인출 계획 ===')
print()
print(f'권장 인출률:    연 {actual_rate*100:.0f}%')
print(f'권장 월 인출액: {monthly_withdrawal:,.0f}원')
print(f'현재 월 생활비: {MONTHLY_EXPENSE:,.0f}원')
print()

# 실제 인출할 금액 (생활비와 권장액 중 작은 금액)
actual_withdrawal = min(monthly_withdrawal, MONTHLY_EXPENSE)
print(f'이번 달 인출액 (확정): {actual_withdrawal:,.0f}원')
print()

# DB 저장
conn = sqlite3.connect(db_path)
cur  = conn.cursor()

# 이번 달 이미 저장됐는지 확인
month_prefix = today[:7]  # YYYY-MM
cur.execute("SELECT id FROM withdrawal_log WHERE date LIKE ?", (f'{month_prefix}%',))
existing = cur.fetchone()

if existing:
    print(f'이번 달({month_prefix}) 인출 기록이 이미 있습니다.')
else:
    cur.execute('''
        INSERT INTO withdrawal_log (date, amount, guardrail_applied, note)
        VALUES (?, ?, ?, ?)
    ''', (today, actual_withdrawal, guardrail_applied,
          f'가드레일 인출율 {actual_rate*100:.0f}% 적용'))
    conn.commit()
    print(f'✅ 인출 기록 저장 완료 ({today})')

conn.close()

=== 이번 달 인출 계획 ===

권장 인출률:    연 4%
권장 월 인출액: 225,167원
현재 월 생활비: 2,500,000원

이번 달 인출액 (확정): 225,167원

✅ 인출 기록 저장 완료 (2026-05-20)


## 6. 인출 이력 조회

In [8]:
conn = sqlite3.connect(db_path)
history = pd.read_sql_query(
    'SELECT date, amount, guardrail_applied, note FROM withdrawal_log ORDER BY date DESC LIMIT 12',
    conn
)
conn.close()

if history.empty:
    print('인출 이력이 없습니다.')
else:
    history['amount'] = history['amount'].apply(lambda x: f'{x:,.0f}원')
    history['guardrail_applied'] = history['guardrail_applied'].map({0: '정상', 1: '하향가드레일'})
    history.columns = ['날짜', '인출금액', '가드레일', '비고']
    print('=== 최근 12개월 인출 이력 ===')
    print(history.to_string(index=False))

=== 최근 12개월 인출 이력 ===
        날짜     인출금액 가드레일             비고
2026-05-20 225,167원   정상 가드레일 인출율 4% 적용


---

## 완료!

- 버킷별 현황과 가드레일 인출법 계산이 완료되었습니다.
- 인출 지속 가능 시뮬레이션으로 장기 안전성을 확인하세요.

**다음 단계**: `03_risk_score.ipynb` — 위험 점수 계산